# Protocolo de equipo

Esta libreta sincroniza tus flujos de n8n con el repositorio de tu equipo. Úsala **cada vez** que traigas
cambios del repositorio (**Pull** en GitHub Desktop) y antes de subir los tuyos (**Commit** y **Push**).

1. Menú **Run → Run All Cells**. **La primera vez te pregunta tu papel y tu subflujo** (la pregunta aparece
   al pie de la primera celda): contéstala y sigue. Lo guarda en `mi-papel.json`, solo en tu equipo.
2. Lee lo que dice cada celda. Si todo salió bien, en GitHub Desktop verás tu archivo cambiado en `flujos/`:
   haz **Commit** y **Push** de **ese** archivo.

Cada quien es dueño de un archivo: nadie edita el archivo de otro. Así no hay conflictos en Git.

**Esta libreta es de todo el equipo: no la subas.** Si GitHub Desktop la muestra como cambiada (correrla
guarda sus resultados), **desmárcala** antes de hacer Commit.

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, "herramientas")
import flujos

CARPETA = Path("flujos")
# La primera vez te pregunta tu papel y tu subflujo y los guarda en mi-papel.json (no se sube a Git)
PAPEL, SUBFLUJOS = flujos.mi_papel()
archivos = (["principal.json"] if PAPEL == "integrador" else []) + [f"{s}.json" for s in SUBFLUJOS]
assert archivos, "Un integrante necesita su subflujo: borra mi-papel.json y corre la libreta otra vez"
MIOS = {}
for archivo in archivos:
    assert (CARPETA / archivo).exists(), f"No está flujos/{archivo}. ¿Ya hiciste Pull en GitHub Desktop?"
    MIOS[json.loads((CARPETA / archivo).read_text(encoding="utf-8"))["name"]] = archivo

n8n = flujos.conectar()
print("Conectado a tu n8n. Lo tuyo:", ", ".join(f"«{n}» ({a})" for n, a in MIOS.items()))

## 1. Traer los flujos del repositorio a tu n8n

Importa todo lo que hay en `flujos/`: primero los subflujos y después el principal, **reconectando cada
llamada por el nombre del subflujo**. Lo tuyo no se reemplaza si ya existe en tu n8n, para no perder lo que
no has exportado.

In [ ]:
for linea in flujos.importar(n8n, CARPETA, excepto=set(MIOS)):
    print(linea)

## 2. Exportar lo tuyo al repositorio

Escribe tu flujo en `flujos/`, sin datos de tu instancia. Después, en GitHub Desktop: **Commit** y **Push**.

In [ ]:
for linea in flujos.exportar(n8n, CARPETA, MIOS):
    print(linea)
print("Ahora, en GitHub Desktop: Commit y Push de lo que cambió en flujos/.")

## 3. Verificar

Revisa que cada llamada a un subflujo apunte de verdad al subflujo que dice su nombre.

In [ ]:
revisadas, problemas = flujos.verificar(n8n)
print(f"{revisadas} llamadas a subflujos revisadas")
for p in problemas:
    print("PROBLEMA:", p)
if not problemas:
    print("Todas apuntan al subflujo correcto.")

## Si algo falla

| Mensaje | Qué hacer |
|---|---|
| `No está flujos/…` | Haz **Pull** en GitHub Desktop. Si eres integrante, revisa que tu subflujo sea el tuyo |
| Te equivocaste de papel o de subflujo | Borra `mi-papel.json` en el panel de archivos de JupyterLab y corre la libreta otra vez: te lo vuelve a preguntar |
| `No encuentro en tu n8n un flujo llamado…` | En tu n8n, el flujo debe llamarse **exactamente** como en el contrato. No le cambies el nombre |
| `…llama a «…», que no está en la carpeta ni en tu n8n` | Su dueño todavía no sube su archivo: pídele que haga Push, y tú, Pull |
| `PROBLEMA: … apunta a un flujo que no existe aquí` | Corre otra vez la libreta completa |
| La libreta no conecta con n8n | Abre la página de estado del entorno: n8n debe estar en verde y «Tu n8n», listo |

**Sin la libreta** (contingencia): en n8n, menú **⋯ → Import... → Import from file...** para cada archivo de `flujos/`,
primero los subflujos. Luego abre en el principal **cada nodo que llama a un subflujo y vuelve a elegirlo
en la lista**, aunque ya muestre el nombre correcto: la referencia puede estar rota sin ninguna advertencia.